<a href="https://colab.research.google.com/github/edwardoughton/spatial_computing/blob/main/5_04_Intro_To_GeoPandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌍 🗺️ **Intro to GeoPandas** 🌍 🗺️

The beauty of `GeoPandas` is that it enables us to manage spatial info using the Python Data Analysis Library: https://pandas.pydata.org

## 🧭 A quick recap on `Pandas`

`Pandas` is a Python package "providing fast, flexible, and expressive data structures designed to make working with “relational” or “labeled” data both easy and intuitive".

It provides us with a range of capabilities:

* DataFrame object for data manipulation with integrated indexing.
* Tools for reading and writing data between in-memory data structures and different file formats.
* Data alignment and integrated handling of missing data.
* Reshaping and pivoting of data sets.
* Label-based slicing, fancy indexing, and subsetting of large data sets.
* Data structure column insertion and deletion.
* Group by engine allowing split-apply-combine operations on data sets.
* Data set merging and joining.
* Hierarchical axis indexing to work with high-dimensional data in a lower-dimensional data structure.
* Time series-functionality: Date range generation[6] and frequency conversions, moving window statistics, moving window linear regressions, date shifting and lagging.
* Provides data filtration.

## 🧭 So what is special about `GeoPandas`?

*`GeoPandas` is a project to add support for geographic data to pandas objects. It currently implements GeoSeries and GeoDataFrame types which are subclasses of pandas.Series and pandas.DataFrame respectively. GeoPandas objects can act on shapely geometry objects and perform geometric operations.*

See the Git repo for more information: https://github.com/geopandas/geopandas

The `GeoPandas` dataframe holds a geometry column which enables cartesian geometry operations (meaning it can interpret pairs of numerical coordinates in space).

The coordinate reference system (crs) can be stored as an attribute on an object, and is automatically set when loading from a file. Objects may be transformed to new coordinate systems with the `.to_crs()` method.

Here we will cover the following basic operations:

* Reading data to a geopandas dataframe
* Manipulating column data
* Creating a new column
* Changing coordinate reference systems
* Writing data to a geopandas dataframe

## Reading data into `GeoPandas`

We can download the Virginia shapefile ZIP automatically over HTTPS. The course repository URL is pinned to a specific commit so everyone uses the same dataset. An internet connection is required.

The next cell downloads and extracts the data into `virginia_data`. Keep the companion files together: `.shp` stores geometry, `.dbf` stores attributes, `.shx` stores the index, and `.prj` describes the coordinate reference system.

**Optional: manual upload in Colab**

Download [virginia.zip](https://raw.githubusercontent.com/edwardoughton/spatial_computing/93d2e72b57741f7cd192f9664d62bafb93c06739/data/virginia.zip) and extract it on your computer. In place of the next cell, run this code in a separate cell and select all the extracted companion files:

```python
from google.colab import files
uploaded = files.upload()
shapefile_path = 'virginia.shp'
```

Then continue with the GeoPandas reading cell below.


In [ ]:
# Example: Download and extract the data automatically over HTTPS
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

data_url = 'https://raw.githubusercontent.com/edwardoughton/spatial_computing/93d2e72b57741f7cd192f9664d62bafb93c06739/data/virginia.zip'
data_dir = Path('virginia_data')
data_dir.mkdir(exist_ok=True)
zip_path = data_dir / 'virginia.zip'
urlretrieve(data_url, zip_path)

with ZipFile(zip_path) as archive:
    archive.extractall(data_dir)

shapefile_path = data_dir / 'virginia.shp'


Import `geopandas` using the conventional short name `gpd`.

Use `.read_file()` with `shapefile_path` to read the downloaded data. GeoPandas also reads its companion files, including the `.prj` that records the source CRS.


In [ ]:
# Example: Read the data and its stored coordinate reference system
import geopandas as gpd

gdf = gpd.read_file(shapefile_path)
if gdf.crs is None:
    raise ValueError('The CRS is missing. Check that the source .prj file is present.')

gdf

What do you notice that is different about this dataframe?

To make it easier to view, let us subset columns of interest.

In [ ]:
# Example: Subsetting our geodataframe
gdf = gdf[['geometry', 'NAME_1', 'NAME_2', 'TYPE_2']]
gdf

## 🧭 Working with Coordinate Reference Systems (CRS)

We need to be able to map data points to precise locations across space. Indeed, this underpins our ability to process and analyze spatial data.

There are hundreds of different types of Coordinate Reference Systems, with many geographic regions specifying their own to enable local consistency and precision.

A Geographic Coordinate System measures locations on Earth in latitude and longitude and is based on either a spherical or ellipsoidal coordinate system.

* Latitude is measured in degrees north or south of the equator.
Longitude is measured in degrees east or west of a prime meridian (a meridian divides a spheroid into two hemispheres).

* See the World Geodetic System (WGS84/EPSG:4326):https://en.wikipedia.org/wiki/World_Geodetic_System

A Projected Coordinate System instead represents Earth locations via a specific map projection using cartesian coordinates (x,y) on a planar (2D) surface.

* This approach maps a curved Earth surface onto a flat 2D plane.
Common units include metric meters and imperial feet.

* See the Universal Transverse Mercator (UTM): https://en.wikipedia.org/wiki/Universal_Transverse_Mercator_coordinate_system

* Or the WGS 84 Pseudo-Mercator: https://epsg.io/3857

We can easily check the current CRS of our data using the `.crs` property, as follows

In [ ]:
# Example: Checking the crs of our dataframe
gdf.crs

Use `.to_crs()` to transform coordinates into another CRS. Below we deliberately use EPSG:3857 (Web Mercator) to explore projection distortion. Its units are meters, but its areas are distorted.

Use `.set_crs()` only to identify a known source CRS when metadata is missing; it does not transform coordinates. Do not guess a CRS from how coordinates look.

In [ ]:
# Example: Convert the geodataframe crs
gdf = gdf.to_crs('epsg:3857')
gdf

What do you now notice about this geodataframe?

We can then easily find the area of a set of geometry polygons, such as the shapes we have here, via the `.area` property.

In [ ]:
# Example: Finding the area of our polygons
gdf['area_km2'] = gdf['geometry'].area / 1e6

### **Task — 🕵️ The case of the expanding picnic blanket**

* Sum `area_km2` to estimate the total area represented by the Virginia polygons in Web Mercator.
* Create `virginia_projected = gdf.to_crs("EPSG:3968")` (NAD83 / Virginia Lambert).
* Recalculate `area_km2` from the transformed geometry, then sum it again. Reprojection does not update an existing numeric area column!
* Compare the totals and explain why a projection changes measured area. EPSG:3968 is designed for Virginia; Web Mercator is primarily for display.
* Reflect: does your dataset include water, and do its polygons overlap? These also affect the meaning of a total.

In [ ]:
# Enter your attempt here


## 🧭 Exporting data via GeoPandas

When needing to export your data, e.g., as shapefiles, there will be some differences to how you previously did this in `Pandas`.

For example, below we will export our data via `.to_file()`.

Make sure you get the correct crs.

In [ ]:
# Example: Export a separate subset and preserve gdf for the next task
export_gdf = gdf[['NAME_2', 'geometry']].copy()
export_gdf.to_file('my_exported_data.shp')

### **Task — 🧺 A Northern Virginia picnic tour**

For this exercise, define our tour as Fairfax, Arlington and Alexandria. Inspect `NAME_2` first and use the exact labels in your dataset; some names may refer to a county and an independent city.

* Use `.isin()` on `NAME_2` to select the tour areas into a new GeoDataFrame. Check the row count and print the selected names.
* Transform to EPSG:3968 and recalculate area in square kilometers.
* Find the sum, mean, median, minimum and maximum area.
* Export the selection to `picnic_tour.shp`. Keep its companion files together.
* **Bonus:** plot your tour with `.plot()` and give the map a cheerful title. Explain why county area alone cannot identify a good picnic site.

In [ ]:
# Enter your attempt here


### 🎯 **Task — 🦆 Map a duck’s dream day out**

These are fictional attractions near Fairfax, with illustrative longitude/latitude coordinates. This task does not need the course shapefile.

* Run the starter cell, then create a map using `duck_stops.plot()`.
* Use `.to_crs(duck_stops.estimate_utm_crs())` to create a local projected copy called `duck_meters`.
* Add a `distance_m` column using `duck_meters.distance(duck_meters.geometry.iloc[0])`. Which stop is furthest from the first?
* Create 250 meter buffers with `duck_meters.geometry.buffer(250)` and plot them.
* Export `duck_meters` to `duck_day_out.geojson` after converting it back to EPSG:4326.
* **Reflect:** these distances are straight lines. What extra information would you need for a safe walking route?

In [ ]:
import geopandas as gpd

duck_stops = gpd.GeoDataFrame(
    {"name": ["Disco Duck Pond", "Cupcake Corner", "Rainbow Reading Nook"]},
    geometry=gpd.points_from_xy(
        [-77.306, -77.302, -77.310],
        [38.830, 38.834, 38.832],
    ),
    crs="EPSG:4326",
)

# x = longitude, y = latitude. Start your map below!


In [ ]:
# Enter your mapping and analysis here


### 🎉 **Before you go…**

Have a go at explaining one new technique to a neighbor. Show them a result you predicted correctly and one thing that surprised you. Add a text cell with your explanation, then save your notebook. Nice work!

### 🤖 **AI usage statment**

I used AI to add in emoji to my classic spatial computing notebooks and make my tasks more whimsical (because I'm not that fun in real life!).